# GR5069 Take Home Exercise 2
### *Xuejing Yan*

#### 1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

- Logic: Since the question asked for pit stop for each race, so I will need to load the pitstops dataset using spark. Then I will need to group raceId and driverId to get the average time for each driver in each race. In order to get the fastest and slowest pit stop time each race, I will group pit stop data by raceId only and get the min/max value. 


In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)


In [0]:
display(df_pitstops)

In [0]:
from pyspark.sql.functions import col

display(df_pitstops.filter(col("milliseconds").isNull()))

df_pitstops = df_pitstops.withColumn(
    "milliseconds",
    col("milliseconds").cast("int")
)

In [0]:
from pyspark.sql.functions import avg, round

driver_race_avg = (
    df_pitstops
    .groupBy("raceId", "driverId")
    .agg(round(avg("milliseconds"), 2).alias("avg_ps"))
)

In [0]:
display(driver_race_avg)

In [0]:
from pyspark.sql.functions import min, max
race_pit_summary = (df_pitstops.groupby("raceId")
                    .agg(min("milliseconds").alias("min_ps"),
                         max("milliseconds").alias("max_ps"))
                    )
display(race_pit_summary)

- I used `groupBy()` function to group raceId and driverId together, then use the function `avg()` to computes the average of all their pit stops in that race. To find the fastest and slowest pit stop in each race overall, I grouped only by `raceId` and used `min("milliseconds")` and `max("milliseconds")`. These return a dataframe with the smallest and largest pit stop times recorded in that race. 

#### 2. Rank order by finishing position the average time spent at the pit stop in each race.

- Logic: In order to get finishing position, i will need to load the `results.csv` and combine with `driver_race_avg` I created from last question. I will join this result with the `results` dataset using `raceId` and `driverId` so that I could associate each driver’s average pit stop time with their finishing position in that race. After that, I rank the drivers within each race based on their finishing position.

In [0]:
df_results = (spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True))

display(df_results)

In [0]:
results_clean = (
    df_results
    .filter(col("position").isNotNull())
    .filter(col("position") != "\\N")
    .withColumn("position", col("position").cast("int"))
)

display(results_clean)

In [0]:

pitstops_position = (
    driver_race_avg
    .join(results_clean, on=["raceId", "driverId"], how="inner")
    .select("raceId", "driverId", "position", "avg_ps")
    .orderBy(col("raceId").asc(), col("position").asc())
)

display(pitstops_position)

- I first loaded the `results.csv` and cleaned it by filtering out rows where `position` was null or equal to `\N`. Because I saw the dataframe have \N value. I then converted `position` to an integer using `.cast("int")` so it could be sorted numerically.

- Next, I joined the average pit stop I computed before table with the cleaned finishing position table using `.join(..., on=["raceId", "driverId"], how="inner")`. I only selected relevant columns, and sorted the output by `raceId` and `position` so that each race is displayed in finishing order.

#### 3.  Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

- Logic: I will need to load the `drivers` dataset and check if there's missing values. I want to keep the exising code and filled the missing code. Since most of the code is the first three letters of driver's surname, so I will follow this rule.

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header = True)

display(df_drivers)

In [0]:
from pyspark.sql.functions import when, trim, upper, substring

drivers_filled_code = (
    df_drivers.withColumn("filled_code",
        when(
            (col("code").isNull()) | (col("code") == "\\N"),
            upper(substring(col("driverRef"), 1, 3))
        ).otherwise(col("code"))
    )
)
    

display(drivers_filled_code)


- In my code, I first loaded the `drivers.csv` file. By looking at the dataframe, I saw many missing values stored as "\N". And I noticed that `driverRef` normalized the driver's surname with no accents and punctuation. To fill the missing codes, I created a new column called `filled_code` using `.withColumn(...)`. Inside that step, I used `when(...) to filter the case where original `code` was null, blank, or equal to "\N". Then I replace the missing values with function ` substring(Fcol("driverRef"), 1, 3)` to extract the first three characters of `driverRef`, and wrapped it with `upper(...)` to get three-letter uppercase format. I also used `otherwise(...)` to preserve all existing codes.

#### 4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

- Logic: To answer this question, I need to combine information from the `drivers.csv` and `results.csv` to map drivers to races. Since every race happened at different time, I need also join with `race.csv` to get the date of the race and compute each driver’s age at the time of the race. I will rank the age to find the oldest and youngest driver in each race. 

In [0]:
from pyspark.sql.functions import current_date, floor, months_between

df_races = (spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header = True))

display(df_races)

In [0]:
df_results = df_results.select("raceId", "driverId").dropDuplicates()

df_driver_race = (
    df_results
    .join(df_drivers, on="driverId", how="left")
    .join(df_races, on="raceId", how="left")
)

display(df_driver_race)


In [0]:
from pyspark.sql.functions import months_between, floor

df_age = df_driver_race.withColumn(
    "Age",
    floor(months_between(col("date"), col("dob")) / 12)
)

display(df_age)

In [0]:
from pyspark.sql.window import Window

youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc())
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc())


In [0]:
from pyspark.sql.functions import row_number

df_ranked = (
    df_age
    .withColumn("youngest_rank", row_number().over(youngest_window))
    .withColumn("oldest_rank", row_number().over(oldest_window))
)


df_age_ranked = (
    df_ranked
    .filter((col("youngest_rank") == 1) | (col("oldest_rank") == 1))
    .select("raceId", "driverId", "forename", "surname", "Age")
    .orderBy("raceId", "Age")
)

display(df_age_ranked)


- In my code, I first loaded the `races.csv` dataset.  I also used `.dropDuplicates()` to ensure that each driver appears only once per race in the `results` dataset, which avoids duplicate rows when joining with other datasets. After that, I joined the datasets together. First, I joined `df_results` with `df_drivers` on `driverId` using a left join, so that each row contains driver-level information such as name and date of birth. Then, I joined the result with `df_races` on `raceId` using a left join, so that each row also includes race-level information such as the race date. I defined age as the number of full years between the race date and the driver’s date of birth. I computed this using the `months_between()` which calculates the difference in months between two dates. I then divided by 12 to convert months into years and applied `floor()` to get an integer age, so this age reflects the driver’s age at the time of the race.
- After creating the Age column, I used window functions to identify the youngest and oldest drivers within each race. I defined two window specifications using `Window.partitionBy("raceId")` to performe ranking separately for each race. I then used `row_number().over(...)` to assign a ranking to each driver within each race for both youngest and oldest criteria. Finally, I filtered the dataset to keep only the rows where either `youngest_rank = 1` or `oldest_rank = 1`.

In [0]:
# Extra Credit: Alternative method
windowSpec = Window.partitionBy("raceId")

age_stats = (
    df_age
    .withColumn("min_age", min("Age").over(windowSpec))
    .withColumn("max_age", max("Age").over(windowSpec))
)


df_age_rank = (
    age_stats
    .filter((col("Age") == col("min_age")) | (col("Age") == col("max_age")))
    .select("raceId", "driverId", "forename", "surname", "Age")
    .orderBy("raceId", "Age")
)

display(df_age_rank)

- As an alternative method, I used aggregation-based window functions instead of ranking. I partitioned the data by `raceId` and used `min()` and `max()` over the window to compute the youngest and oldest ages within each race. I then filtered the dataset to keep rows where the driver’s age matches either the minimum or maximum age for that race. 


#### 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver.

- Logic: To determine the number of podiums a driver has "at any given race," I need to count their career finishes for 1st, 2nd, and 3rd place up to that specific date. I will join the `results` dataset with `races` to sort the races chronologically. Then I will need to create three indicator columns to store their podiums status. The I will compute cumulative sums of these indicator columns. 

In [0]:
df_race_result = results_clean.join(df_races.select("raceId", "date"), on="raceId", how="left")

df_podium = (
    df_race_result
    .withColumn("win", when(col("position") == 1, 1).otherwise(0))
    .withColumn("second", when(col("position") == 2, 1).otherwise(0))
    .withColumn("third", when(col("position") == 3, 1).otherwise(0))
)

display(df_podium)

In [0]:
from pyspark.sql.functions import sum
windowSpec = (Window
    .partitionBy("driverId")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_podium_counts = (
    df_podium
    .withColumn("win_count", sum("win").over(windowSpec))
    .withColumn("second_count", sum("second").over(windowSpec))
    .withColumn("third_count", sum("third").over(windowSpec))
)

display(df_podium_counts)

In [0]:
df_podium_counts = (
    df_podium_counts
    .select("raceId", "driverId", "win_count", "second_count", "third_count")
    .orderBy("driverId")
)

display(df_podium_counts)

- I used the `clean_result` dataset and joined with the `races` dataset. Next, I created three indicator columns using `when(...)`: `win`, `second`, and `third`. Each column takes the value 1 if the driver finished in that position, and 0 otherwise. 
To compute cumulative podium counts, I defined a window partitioned by `driverId` and ordered by race date. I used `rowsBetween(Window.unboundedPreceding, Window.currentRow)` to make sure that the sum includes all previous races up to the current one. I then applied `sum(...) over the window` to get the running totals for wins, second places, and third places. 

#### 6. Continue exploring the data by answering your own question.
- What are the most common reasons drivers fail to finish a race?

- Logic: I will use the `status` dataset to get race outcome then joined with  `results` dataset. Then, I filtered the data to keep only those who did not finish. 
After that, I can group the data by status description and count how frequently each type of failure occurred. Finally, I will sort the results in descending order to identify the most common reasons for not finishing a race.

In [0]:
df_status = spark.read.csv("/Volumes/gr5069/raw/f1_data/status.csv", header=True)

df_result_status = (results_clean.join(df_status, on="statusId", how="left"))

df_dnf = (df_result_status.filter(col("status") != "Finished"))


In [0]:

from pyspark.sql.functions import count

dnf_counts = (
    df_dnf
    .groupBy("status")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
)

display(dnf_counts)

- I loaded the `status` dataset then joined this dataset with the `results` dataset using `statusId`. Next, I filtered the dataset to exclude rows where the status is “Finished”, since I am specifically interested in drivers who did not complete the race. After filtering, I grouped the data by the `status` column and used the `count()` function to calculate how frequently each failure type occurs. Finally, I sorted the results in descending order to identify the most common reasons drivers fail to finish races.
